In [130]:
import warnings
warnings.filterwarnings('ignore')

In [131]:
import pandas as pd
import numpy as np
import pyreadr
import joblib
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import plotly.express as px

from tqdm import tqdm

from dash import Dash, html, dcc, dash_table, Input, Output, callback
import dash_bootstrap_components as dbc

In [132]:
pd.set_option('display.max_columns', 100)

In [133]:
dem_uncont = 34
rep_uncont = 31

In [134]:
seat_sims = pyreadr.read_r('model_output/tot_seats_sims.RDS')[None]
seat_sims = seat_sims.rename({None: 'seats'}, axis=1)
seat_sims['seats'] = seat_sims['seats'].map(lambda x: x + dem_uncont)
seat_sims['winner'] = seat_sims['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims

,seats,winner
0,53,Democrats
1,55,Democrats
2,48,Republicans
3,53,Democrats
4,48,Republicans
...,...,...
7995,51,Democrats
7996,51,Democrats
7997,53,Democrats
7998,51,Democrats


In [135]:
sim_counts = seat_sims.groupby(['seats']).count().reset_index().rename({'winner': 'count'}, axis=1)
n_sims = seat_sims.shape[0]
sim_counts['pct'] = sim_counts['count'] / n_sims * 100
sim_counts['winner'] = sim_counts['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims = pd.merge(left=seat_sims.drop(['winner'], axis=1), right=sim_counts, on='seats', how='left')
def get_desc(winner, pct, seats):
    return f'{winner} wins {seats if seats >= 51 else (100-seats)} seats in {pct:.2f}% of simulations'
#seat_sims['desc'] = seat_sims[['winner', 'pct', 'seats']].apply(lambda x: get_desc(x['winner'], x['pct'], x['seats']), axis=1)
sim_counts.head()

,seats,count,pct,winner
0,36,1,0.0125,Republicans
1,38,1,0.0125,Republicans
2,40,2,0.0250,Republicans
3,41,3,0.0375,Republicans
4,42,12,0.1500,Republicans


In [136]:
np.unique(seat_sims['seats']).shape[0]

27

In [137]:
sims_hist = px.histogram(seat_sims, x='seats', nbins=np.unique(seat_sims['seats']).shape[0]*2, color='winner',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},
                         labels={'seats':'Seats won by Democrats', 'winner': 'Winner'}, template='plotly_white')
sims_hist.update_traces(showlegend=False)
sims_hist.add_vline(x=50.5, line_width=1, line_color='black', annotation_text='51 seats required for Dem majority', 
                    annotation_position='top right')
sims_hist

In [138]:
output_ts = pd.read_csv('model_output/output_over_time.csv')
output_ts.head()

,date,y,geo,type
0,2026-09-27,51.348375,US Senate,seats
1,2026-09-27,58.600000,US Senate,chance
2,2026-09-27,3.816193,US Senate,seats_sd
3,2026-09-27,38.771876,AL,y_pred
4,2026-09-27,3.537242,AL,y_pred_sd


In [139]:
chance_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'chance')]
chance_over_time['rep_chance'] = chance_over_time['y'].map(lambda x: 100 - x)
chance_over_time = chance_over_time.rename({'y': 'Democrats', 'rep_chance': 'Republicans'}, axis=1)
chance_time_ser = px.line(chance_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
chance_time_ser.update_traces(hovertemplate="%{y:.1f}%")
chance_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Win Probability (%)',
    title=dict(text="Senate Win Probability Over Time"),
    hovermode="x",
    showlegend=False
)
chance_time_ser

In [140]:
seats_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'seats')]
seats_over_time['rep_seats'] = seats_over_time['y'].map(lambda x: 100 - x)
seats_over_time = seats_over_time.rename({'y': 'Democrats', 'rep_seats': 'Republicans'}, axis=1)
seats_time_ser = px.line(seats_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                        color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
seats_time_ser.update_traces(hovertemplate="%{y:.1f}")
seats_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Average Seats Over All Simulations',
    title=dict(text="Projected Seats Over Time"),
    hovermode="x",
    showlegend=False
)
seats_time_ser

In [141]:
joblib.dump(sims_hist, 'display_data/sims_histogram.pkl')
joblib.dump(chance_time_ser, 'display_data/chance_time_ser.pkl')
joblib.dump(seats_time_ser, 'display_data/seats_time_ser.pkl')

['display_data/seats_time_ser.pkl']

In [142]:
# posterior prediction
post = pyreadr.read_r('model_output/labeled_posterior.RDS')[None]
post_untransp = post.copy()

In [143]:
post = post.T
post.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,7950,7951,7952,7953,7954,7955,7956,7957,7958,7959,7960,7961,7962,7963,7964,7965,7966,7967,7968,7969,7970,7971,7972,7973,7974,7975,7976,7977,7978,7979,7980,7981,7982,7983,7984,7985,7986,7987,7988,7989,7990,7991,7992,7993,7994,7995,7996,7997,7998,7999
AL,-5.213885,-11.761162,-9.697229,-9.973844,-7.784955,-11.259019,-9.769082,-3.532153,-10.009396,-15.463018,1.049503,-11.050558,-8.828489,-9.488983,-10.196361,-7.593852,-7.806500,-10.332856,-7.924756,-5.386282,-10.663966,-7.012049,-5.001094,-7.224196,-6.704385,-12.578416,-12.030286,-5.974535,-11.025873,-11.205035,-8.591767,-9.760836,-9.473277,-6.005281,-14.097690,-13.076076,-5.880829,-8.913972,-6.596049,-10.677985,-8.550136,-6.789102,-11.878386,-8.726803,-10.872768,-11.436095,-10.939550,-12.997750,-5.965685,-9.503456,...,-9.664731,-9.020589,-7.819853,-13.113059,-5.066432,-9.738230,-10.835607,-5.719788,-10.191272,-6.238422,-14.242859,-14.245033,-8.501655,-5.177330,-15.527117,-5.121919,-14.480399,-1.031338,-9.950412,-6.872859,-9.070243,-7.754655,-2.683704,-19.284378,-1.328132,-11.852197,-9.497205,-11.528470,-5.785771,-10.523507,-8.766592,-10.811054,-8.921799,-5.999954,-14.747196,-10.325896,-8.947331,-16.776495,-7.940479,-8.061516,-8.921921,-12.139421,-8.610144,-8.497722,-10.154579,-11.014062,-13.026538,-7.890681,-10.679805,-10.190525
AK,0.612346,5.717614,-0.031717,5.631773,-1.004881,4.682787,4.977483,5.269189,-0.517636,-5.589522,10.953966,0.411172,-0.907565,6.639151,-0.713430,3.108963,5.699605,4.749738,-0.331185,2.900484,6.969569,4.253090,2.380662,5.208009,5.011421,1.382282,2.410226,3.687449,-0.875743,1.537822,3.428450,5.775728,7.094391,8.377505,2.834467,-0.251086,3.687020,0.074980,-0.969204,3.242617,-1.222024,3.750881,7.103642,4.089460,2.704990,-1.681605,1.025501,3.131170,2.138931,4.486953,...,4.005598,2.889898,10.076592,-0.230888,7.620264,2.697083,1.378852,3.692516,-1.416625,10.564120,2.434801,-0.876719,7.082141,3.378609,3.803858,1.610861,4.859181,2.233532,-0.503795,7.351568,1.818267,3.439659,8.300425,0.254099,3.910083,-2.160468,6.562280,4.856947,3.833208,0.855767,-3.095928,5.195825,-0.631353,6.037373,1.706199,3.114833,3.340695,7.215622,4.137891,4.053272,0.054509,4.276467,3.969430,0.261507,3.934995,1.061370,1.777866,0.732016,3.531726,1.752006
AR,-10.715973,-12.618259,-8.992539,-11.724732,-10.432890,-8.611054,-9.721514,-6.261093,-14.669968,-14.733259,-1.672887,-12.331302,-13.227551,-7.857998,-12.707785,-11.268400,-10.620095,-9.089300,-11.383529,-7.810878,-11.842100,-11.376695,-7.591040,-7.841899,-12.075883,-9.339373,-11.827783,-8.509231,-11.809112,-11.628103,-8.993054,-9.682349,-7.181601,-7.968930,-13.999569,-13.414092,-6.849205,-10.242565,-11.800414,-10.876518,-10.846130,-6.877657,-12.812815,-14.439429,-13.147016,-11.371824,-9.562748,-14.510601,-11.204917,-12.484261,...,-13.434509,-11.836720,-8.513728,-14.090129,-4.299725,-12.769265,-13.376801,-12.041449,-10.755566,-4.623570,-15.115502,-12.679728,-8.712582,-8.161006,-14.018339,-5.241928,-16.029640,-1.766034,-9.815535,-8.968800,-12.254606,-6.857986,-5.257479,-16.076776,-3.750829,-17.047898,-9.777613,-16.524033,-5.850256,-10.235093,-9.681351,-11.304355,-10.218615,-3.119714,-17.918956,-10.249578,-8.306927,-12.937494,-6.475532,-12.730996,-11.641840,-11.202710,-9.244264,-7.481353,-11.662535,-14.087876,-17.633894,-11.825852,-8.077864,-11.271660
CO,11.226725,12.012826,7.088213,13.934999,8.688065,12.383422,14.174138,13.073763,10.639500,6.968953,13.129749,13.622843,10.737693,15.996512,7.753444,15.482422,13.533261,11.625599,6.633162,5.903605,12.757912,18.523407,9.726477,16.256633,13.509033,11.306368,11.133334,19.881731,7.551049,13.354098,13.912435,10.294465,9.213104,10.305177,11.598316,5.638693,16.147729,11.027974,15.252341,9.031596,12.887590,11.859133,11.491852,13.423646,10.419738,11.556005,7.478638,10.942282,12.221983,9.360649,...,13.643233,8.463388,17.240432,10.073251,13.048247,6.742136,11.478113,

In [144]:
sim_corr = post_untransp.corr()

In [145]:
sim_corr.head()

,AL,AK,AR,CO,DE,FL,GA,ID,IL,IA,KS,KY,LA,ME,MA,MI,MN,MS,MT,NE,NH,NJ,NM,NC,OH,OK,OR,RI,SC,SD,TN,TX,VA,WV,WY
AL,1.000000,0.516398,0.738692,0.490218,0.497037,0.698721,0.529468,0.735341,0.704366,0.739355,0.732618,0.540740,0.749325,0.732396,0.521050,0.729616,0.705718,0.742820,0.693159,0.721324,0.498627,0.506424,0.504874,0.703090,0.505484,0.756697,0.506923,0.487174,0.487869,0.512250,0.556076,0.694357,0.504739,0.739132,0.737196
AK,0.516398,1.000000,0.518617,0.521969,0.526124,0.507767,0.558281,0.526941,0.504626,0.521958,0.521084,0.568661,0.525537,0.523552,0.561288,0.513178,0.499311,0.518795,0.495123,0.517270,0.561158,0.540128,0.554083,0.498611,0.553238,0.522768,0.548076,0.549655,0.539208,0.543446,0.598447,0.506760,0.544534,0.521488,0.520727
AR,0.738692,0.518617,1.000000,0.487270,0.499832,0.703745,0.526767,0.737937,0.699151,0.741198,0.737037,0.537526,0.742023,0.733307,0.523047,0.728663,0.699474,0.726033,0.693227,0.721270,0.508796,0.500907,0.501528,0.704667,0.505029,0.744103,0.501886,0.494528,0.495270,0.514527,0.556793,0.699818,0.499125,0.735659,0.734966
CO,0.490218,0.521969,0.487270,1.000000,0.708932,0.651351,0.550560,0.497623,0.653912,0.502349,0.485953,0.549366,0.503839,0.486045,0.537781,0.487109,0.642724,0.493566,0.652121,0.475729,0.710066,0.715783,0.719707,0.647074,0.705747,0.499779,0.710415,0.700509,0.697629,0.527919,0.571409,0.654502,0.703957,0.496891,0.494078
DE,0.497037,0.526124,0.499832,0.708932,1.000000,0.665456,0.543190,0.504432,0.664555,0.507233,0.498607,0.547880,0.503291,0.502187,0.549029,0.501463,0.664071,0.499343,0.662647,0.488025,0.718517,0.725909,0.728601,0.661215,0.715595,0.515778,0.709991,0.711506,0.707591,0.519298,0.577976,0.658394,0.719022,0.499507,0.509408


In [146]:
post.shape

(35, 8000)

In [147]:
def get_tipping_point(sim):
    """
    :param sim: Series representing one posterior draw or "simulation"
    :type sim: pd.Series
    """
    seats_won_dem = np.sum(sim > 0)
    if seats_won_dem >= 51 - dem_uncont: # Democrats win House in this draw
        sim = sim.sort_values(ascending=True)
        won_seats = sim[sim > 0]
        seat_margin = seats_won_dem - (51 - dem_uncont)
    else: # Republicans win House in this draw
        sim = sim.sort_values(ascending=False)
        won_seats = sim[sim < 0]
        seat_margin = (100 - seats_won_dem - dem_uncont - rep_uncont) - (50 - rep_uncont)
    return won_seats.iloc[seat_margin - 1], won_seats.index[seat_margin - 1]

In [148]:
tipping_points = np.array([]) # tipping point for *each sim*

for i in tqdm(range(post.shape[1])):
    sim = post.iloc[:, i]
    _, tp_seat = get_tipping_point(sim)
    tipping_points = np.append(tipping_points, tp_seat)

tipping_points

100%|████████████████████████████████████████████████████████████████████████████| 8000/8000 [00:05<00:00, 1486.79it/s]


array(['NE', 'MI', 'MI', ..., 'AK', 'RI', 'AK'],
      shape=(8000,), dtype='<U32')

In [149]:
data = pd.read_csv('../../model_output/senate_predictions.csv')
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.366961,0,0,38.246192,1.880871,49.145126,1.880871,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.880871,1.371448,10.898934,-21.259481,-48.013273,0,-96.026546,41.016300,3.547635,0.5750,1,34.164062,48.141732
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.366961,0,1,50.368983,3.386083,48.179944,3.386083,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,3.386083,1.840131,-2.189039,-4.543644,35.161107,-1,70.322215,53.205310,3.399151,82.7625,2,46.578144,59.995592
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.366961,0,1,39.806386,1.200641,52.528412,1.200641,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.200641,1.095738,12.722026,-22.254082,2.405409,-1,4.810817,39.652077,3.571361,0.1750,3,32.524132,46.605249
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.366961,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.000000,0.000000,0.000000,20.296097,48.015860,1,96.031719,61.583708,3.503933,99.9500,4,54.797665,68.483097
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.366961,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.389350,47.316437,1,94.632875,63.059875,3.504356,99.9875,5,56.222433,70.011351


In [150]:
data['tipping_point_prob'] = data['state_po'].map(lambda x: np.mean(tipping_points == x) * 100)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.366961,0,0,38.246192,1.880871,49.145126,1.880871,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.880871,1.371448,10.898934,-21.259481,-48.013273,0,-96.026546,41.016300,3.547635,0.5750,1,34.164062,48.141732,0.0000
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.366961,0,1,50.368983,3.386083,48.179944,3.386083,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,3.386083,1.840131,-2.189039,-4.543644,35.161107,-1,70.322215,53.205310,3.399151,82.7625,2,46.578144,59.995592,5.5375
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.366961,0,1,39.806386,1.200641,52.528412,1.200641,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.200641,1.095738,12.722026,-22.254082,2.405409,-1,4.810817,39.652077,3.571361,0.1750,3,32.524132,46.605249,0.0000
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.366961,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.000000,0.000000,0.000000,20.296097,48.015860,1,96.031719,61.583708,3.503933,99.9500,4,54.797665,68.483097,0.0875
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.366961,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.389350,47.316437,1,94.632875,63.059875,3.504356,99.9875,5,56.222433,70.011351,0.1875


In [151]:
data.sort_values('tipping_point_prob', ascending=False).head(5)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
15,15,Michigan,Abdul El-Sayed,Mike J. Rogers,False,False,D,MI,"EL-SAYED, ABDUL","ROGERS, MICHAEL J",4.719966e+07,15432313.28,6.263198e+07,75.360329,24.639671,-0.192456,49.278763,77.709102,2.333008,4.148895,12.968600,0.319413,32.445883,7404258,2673073,10077331,73.474395,26.525605,0.987401,-8.366961,0,0,48.691399,9.738105,45.053031,9.738105,East North Central,2026,0.0,0.00000,0.00000,2,Senate,5679.179195,9.738105,3.120594,-3.638368,7.982049,25.360329,0,50.720658,50.881122,3.539413,59.4750,16,44.099365,57.885463,11.2625
31,31,Texas,James Talarico,Ken Paxton,False,False,R,TX,"TALARICO, JAMES","PAXTON, WARREN KENNETH JR.",1.397766e+08,20997954.85,1.607746e+08,86.939504,13.060496,-5.916874,43.060966,48.286718,4.530331,31.642337,13.173480,0.180615,33.823334,24400697,4744808,29145505,83.720275,16.279725,0.987401,-8.366961,0,0,49.185592,13.073109,46.334907,13.073109,West South Central,2026,0.0,0.25284,-0.25284,6,Senate,7558.477337,13.073109,3.615675,-2.850685,-3.466787,36.939504,0,73.879008,51.291993,3.645951,63.8875,32,44.172314,58.554928,10.1750
13,13,Maine,Troy Jackson,Susan Collins,False,True,R,ME,"JACKSON, TROY DALE","COLLINS, SUSAN M.",2.397712e+07,15299978.61,3.927710e+07,61.046056,38.953944,3.820912,53.544306,94.249770,0.897865,1.689465,0.930085,0.342309,36.189622,526309,836050,1362359,38.632181,61.367819,0.987401,-8.366961,0,1,49.530793,7.368513,46.124771,7.368513,New England,2026,0.0,0.00000,0.00000,3,Senate,3726.620896,7.368513,2.714501,-3.406022,16.008785,11.046056,-1,22.092111,50.045235,3.577689,50.3000,14,43.043936,57.031036,9.5750
24,24,Ohio,Sherrod Brown,Jon Husted,False,True,R,OH,"BROWN, SHERROD","HUSTED, JON",5.786194e+07,9843192.51,6.770513e+07,85.461675,14.538325,-5.266554,44.343385,80.925356,1.708882,3.291326,11.486903,0.076606,31.487184,9001099,2798349,11799448,76.284069,23.715931,0.987401,-8.366961,0,1,48.526154,3.005728,45.230828,3.005728,East North Central,2026,0.0,0.00000,0.00000,2,Senate,7303.697897,3.005728,1.733704,-3.295326,-2.166147,35.461675,-1,70.923350,50.675196,3.467472,57.7125,25,43.781174,57.538453,9.1125
10,10,Kansas,Adam Hamilton,Roger Marshall,False,True,R,KS,"HAMILTON, ADAM","MARSHALL, ROGER W",1.652601e+07,2514153.97,1.904016e+07,86.795522,13.204478,-8.032578,41.794263,80.203805,2.235100,8.765322,5.263831,0.443291,35.603269,2124059,813821,2937880,72.299039,27.700961,0.987401,-8.366961,0,1,48.881588,4.574156,46.259309,4.574156,West North Central,2026,0.0,0.00000,0.00000,2,Senate,7533.462650,4.574156,2.138728,-2.622279,-7.698195,36.795522,-1,73.591044,49.504748,3.532331,44.0000,11,42.595146,56.513332,8.7250


In [152]:
def get_rating(dem_chance):
    if dem_chance > 100:
        raise ValueError('Invalid win chance.')
    if dem_chance > 95:
        return 'Safe D'
    elif dem_chance >= 90:
        return 'Very Likely D'
    elif dem_chance >= 75:
        return 'Likely D'
    elif dem_chance >= 65:
        return 'Lean D'
    elif dem_chance >= 60:
        return 'Tilt D'
    elif dem_chance >= 40:
        return 'Tossup'
    elif dem_chance >= 35:
        return 'Tilt R'
    elif dem_chance >= 25:
        return 'Lean R'
    elif dem_chance >= 10:
        return 'Likely R'
    elif dem_chance >= 5:
        return 'Very Likely R'
    else:
        return 'Safe R'

def get_matchup(dem_cand, rep_cand, dem_inc_any, rep_inc_any):
    indie_d = ['Dan Osborn', 'Brian Bengs', 'Todd Achilles', 'Seth Bodnar']
    indie_r = []
    
    if dem_cand in indie_d:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (I)'
        dem_color = '#792ba6'
    else:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (D)'
        dem_color = '#366bbf'
    
    if rep_cand in indie_r:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (I)'
        rep_color = '#792ba6'
    else:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (R)'
        rep_color = '#e63929'

    return f'<b style="color:{dem_color};">' + dem_lab + f'</b> vs <b style="color:{rep_color};">' + rep_lab + '</b>'

In [153]:
data['rating'] = data['chance'].map(lambda x: get_rating(x))
for party in ['dem', 'rep']:
    data[f'{party}_cand'] = data[f'{party}_cand'].map(lambda x: 'TBD' if x[:3] == 'TBD' else x)
data['matchup'] = data[['dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand'],
                                                                                                          x['dem_inc_any'], x['rep_inc_any']), axis=1)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.366961,0,0,38.246192,1.880871,49.145126,1.880871,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.880871,1.371448,10.898934,-21.259481,-48.013273,0,-96.026546,41.016300,3.547635,0.5750,1,34.164062,48.141732,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b..."
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.366961,0,1,50.368983,3.386083,48.179944,3.386083,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,3.386083,1.840131,-2.189039,-4.543644,35.161107,-1,70.322215,53.205310,3.399151,82.7625,2,46.578144,59.995592,5.5375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>..."
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.366961,0,1,39.806386,1.200641,52.528412,1.200641,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.200641,1.095738,12.722026,-22.254082,2.405409,-1,4.810817,39.652077,3.571361,0.1750,3,32.524132,46.605249,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<..."
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.366961,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.000000,0.000000,0.000000,20.296097,48.015860,1,96.031719,61.583708,3.503933,99.9500,4,54.797665,68.483097,0.0875,Safe D,"<b style=""color:#366bbf;"">John Hickenlooper* (..."
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.366961,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.389350,47.316437,1,94.632875,63.059875,3.504356,99.9875,5,56.222433,70.011351,0.1875,Safe D,"<b style=""color:#366bbf;"">Chris Coons* (D)</b>..."


In [154]:
data['projected_winner'] = data['chance'].map(lambda x: 'D' if x > 50 else 'R')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.366961,0,0,38.246192,1.880871,49.145126,1.880871,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.880871,1.371448,10.898934,-21.259481,-48.013273,0,-96.026546,41.016300,3.547635,0.5750,1,34.164062,48.141732,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.366961,0,1,50.368983,3.386083,48.179944,3.386083,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.386083,1.840131,-2.189039,-4.543644,35.161107,-1,70.322215,53.205310,3.399151,82.7625,2,46.578144,59.995592,5.5375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.366961,0,1,39.806386,1.200641,52.528412,1.200641,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.200641,1.095738,12.722026,-22.254082,2.405409,-1,4.810817,39.652077,3.571361,0.1750,3,32.524132,46.605249,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R


In [155]:
data['hold'] = data['projected_winner'] ==  data['current_party']
data['flip'] = data['hold'].map(lambda x: not x)
data['flip_indic'] = data['flip'].map(lambda x: 'Flip' if x else '')
#data['flip'] = data['flip'].map(lambda x: 'Yes' if x else 'No')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.366961,0,0,38.246192,1.880871,49.145126,1.880871,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.880871,1.371448,10.898934,-21.259481,-48.013273,0,-96.026546,41.01630,3.547635,0.5750,1,34.164062,48.141732,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.366961,0,1,50.368983,3.386083,48.179944,3.386083,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.386083,1.840131,-2.189039,-4.543644,35.161107,-1,70.322215,53.20531,3.399151,82.7625,2,46.578144,59.995592,5.5375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip


In [156]:
data['projected_2p_margin'] = data['y_pred'].map(lambda y_pred: f'D+{y_pred - (100-y_pred):.1f}' if y_pred > 50 else f'R+{(100-y_pred) - y_pred:.1f}')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.366961,0,0,38.246192,1.880871,49.145126,1.880871,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.880871,1.371448,10.898934,-21.259481,-48.013273,0,-96.026546,41.016300,3.547635,0.5750,1,34.164062,48.141732,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.0
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.366961,0,1,50.368983,3.386083,48.179944,3.386083,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.386083,1.840131,-2.189039,-4.543644,35.161107,-1,70.322215,53.205310,3.399151,82.7625,2,46.578144,59.995592,5.5375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.4
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.366961,0,1,39.806386,1.200641,52.528412,1.200641,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.200641,1.095738,12.722026,-22.254082,2.405409,-1,4.810817,39.652077,3.571361,0.1750,3,32.524132,46.605249,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.7


In [157]:
data['rep_chance'] = data['chance'].map(lambda x: 100 - x)
data['rounded_dem_chance'] = data['chance'].map(lambda x: np.round(x, 1))
data['rounded_rep_chance'] = data['rep_chance'].map(lambda x: np.round(x, 1))
data['disp_dem_chance'] = data['rounded_dem_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data['disp_rep_chance'] = data['rounded_rep_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.366961,0,0,38.246192,1.880871,49.145126,1.880871,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.880871,1.371448,10.898934,-21.259481,-48.013273,0,-96.026546,41.016300,3.547635,0.5750,1,34.164062,48.141732,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.0,99.4250,0.6,99.4,<1%,>99%
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.366961,0,1,50.368983,3.386083,48.179944,3.386083,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.386083,1.840131,-2.189039,-4.543644,35.161107,-1,70.322215,53.205310,3.399151,82.7625,2,46.578144,59.995592,5.5375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.4,17.2375,82.8,17.2,82.8%,17.2%
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.366961,0,1,39.806386,1.200641,52.528412,1.200641,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.200641,1.095738,12.722026,-22.254082,2.405409,-1,4.810817,39.652077,3.571361,0.1750,3,32.524132,46.605249,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.7,99.8250,0.2,99.8,<1%,>99%


In [158]:
data['swing_24_to_26'] = data['y_pred'].astype(float).map(lambda x: x - (100 - x)) - data['dem_2p_24'].astype(float).map(lambda x: x - (100 - x))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.366961,0,0,38.246192,1.880871,49.145126,1.880871,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.880871,1.371448,10.898934,-21.259481,-48.013273,0,-96.026546,41.016300,3.547635,0.5750,1,34.164062,48.141732,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.0,99.4250,0.6,99.4,<1%,>99%,12.913823
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.366961,0,1,50.368983,3.386083,48.179944,3.386083,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.386083,1.840131,-2.189039,-4.543644,35.161107,-1,70.322215,53.205310,3.399151,82.7625,2,46.578144,59.995592,5.5375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.4,17.2375,82.8,17.2,82.8%,17.2%,20.103745
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.366961,0,1,39.806386,1.200641,52.528412,1.200641,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.200641,1.095738,12.722026,-22.254082,2.405409,-1,4.810817,39.652077,3.571361,0.1750,3,32.524132,46.605249,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.7,99.8250,0.2,99.8,<1%,>99%,10.644141


In [159]:
data['disp_24_to_26_swing'] = data['swing_24_to_26'].map(lambda x: f'D+{x:.1f}' if x > 0 else f'R+{abs(x):.1f}')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26,disp_24_to_26_swing
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.366961,0,0,38.246192,1.880871,49.145126,1.880871,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.880871,1.371448,10.898934,-21.259481,-48.013273,0,-96.026546,41.01630,3.547635,0.5750,1,34.164062,48.141732,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.0,99.4250,0.6,99.4,<1%,>99%,12.913823,D+12.9
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.366961,0,1,50.368983,3.386083,48.179944,3.386083,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.386083,1.840131,-2.189039,-4.543644,35.161107,-1,70.322215,53.20531,3.399151,82.7625,2,46.578144,59.995592,5.5375,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.4,17.2375,82.8,17.2,82.8%,17.2%,20.103745,D+20.1


In [160]:
tab_data = data[['state', 'dem_cand', 'rep_cand', 'rating', 'disp_dem_chance', 'disp_rep_chance', 'projected_2p_margin', 'disp_24_to_26_swing',
                 'tipping_point_prob']]
for party in ['dem', 'rep']:
    tab_data[f'disp_{party}_chance'] = tab_data[f'disp_{party}_chance'].map(lambda x: f'<p style="color:{'blue' if party == 'dem' else 'red'};">{x}</p>')
    tab_data[f'{party}_cand'] = tab_data[f'{party}_cand'].map(lambda x: f'{x} (Ind)' if x in ['Dan Osborn', 'Brian Bengs', 
                                                                                              'Todd Achilles', 'Seth Bodnar'] else x)
tab_data = tab_data.rename({
    'state': 'State',
    'dem_cand': 'Democrat',
    'rep_cand': 'Republican',
    'rating': 'Rating',
    'disp_dem_chance': 'Dem Chance',
    'disp_rep_chance': 'Rep Chance',
    'projected_2p_margin': 'Projected Margin',
    'tipping_point_prob': 'Tipping Point Chance',
    'disp_24_to_26_swing': 'Swing from 2024 Pres'
}, axis=1)
tab_data.head()

,State,Democrat,Republican,Rating,Dem Chance,Rep Chance,Projected Margin,Swing from 2024 Pres,Tipping Point Chance
0,Alabama,Everett Weiss,Barry Moore,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+18.0,D+12.9,0.0000
1,Alaska,Mary Peltola,Dan S. Sullivan,Likely D,"<p style=""color:blue;"">82.8%</p>","<p style=""color:red;"">17.2%</p>",D+6.4,D+20.1,5.5375
2,Arkansas,Hallie Shoffner,Tom Cotton,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+20.7,D+10.6,0.0000
3,Colorado,John Hickenlooper,Mark Baisley,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+23.2,D+11.9,0.0875
4,Delaware,Chris Coons,Michael Katz,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+26.1,D+11.2,0.1875


In [161]:
data['rep_pred'] = data['y_pred'].map(lambda x: 100 - x)
data['projected_2p_margin_number'] = data['rep_pred'] - data['y_pred']
data['proj_seat_lean'] = data['projected_2p_margin_number'] - data['generic_ballot_avg']
sv_bias = float(data.sort_values('tipping_point_prob', ascending=False).reset_index().loc[0, 'proj_seat_lean']) # Seats-votes bias, + = R, - = D
sv_bias

6.604716286897645

In [162]:
topline_stats = pd.read_csv('display_data/topline_stats.csv')
if 'sv_bias' in topline_stats['vars'].values:
    topline_stats = topline_stats[topline_stats['vars'] != 'sv_bias']
topline_stats = pd.concat([topline_stats, pd.DataFrame({'vars': ['sv_bias'], 'x': sv_bias})], axis=0)
topline_stats.to_csv('display_data/topline_stats.csv')
topline_stats

,vars,x
0,means_seats_tot,51.927375
1,chamber_win_chance,62.200000
0,sv_bias,6.604716


In [163]:
mean_seats_tot = topline_stats[topline_stats['vars'] == 'means_seats_tot']['x'].values[0]
chamber_win_chance = topline_stats[topline_stats['vars'] == 'chamber_win_chance']['x'].values[0]

In [164]:
chances = topline_stats[topline_stats['vars'] == 'chamber_win_chance'].set_index(['vars']).T
chances['Republicans'] = chances['chamber_win_chance'].map(lambda x: 100 - x)
chances = chances.rename({'chamber_win_chance': 'Democrats'}, axis=1).T.reset_index()
chances = chances.rename({'x': 'Win Probability'}, axis=1)
seats = topline_stats[topline_stats['vars'] == 'means_seats_tot'].set_index(['vars']).T
seats['Republicans'] = seats['means_seats_tot'].map(lambda x: 100 - x)
seats = seats.rename({'means_seats_tot': 'Democrats'}, axis=1).T.reset_index().rename({'x': 'Seat Share'}, axis=1)
summary_stats = pd.merge(left=chances, right=seats, on='vars', how='inner').rename({'vars': 'Party'}, axis=1)
summary_stats.to_csv('display_data/summary_stats.csv')
summary_stats

,Party,Win Probability,Seat Share
0,Democrats,62.2,51.927375
1,Republicans,37.8,48.072625


In [165]:
incl_cols = ['state', 'dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any', 'rating', 'chance', 'disp_dem_chance',
            'disp_rep_chance', 'projected_2p_margin', 'y_pred', 'flip_indic', 'matchup', 'swing_24_to_26', 'disp_24_to_26_swing']
disp_data = data[incl_cols]

In [166]:
rating_ts = output_ts[output_ts['type'] == 'chance']
rating_ts['rating'] = rating_ts['y'].map(get_rating)
rating_ts['date'] = pd.to_datetime(rating_ts['date'])
rating_ts.head()

,date,y,geo,type,rating
1,2026-09-27,58.6000,US Senate,chance,Tossup
5,2026-09-27,0.0500,AL,chance,Safe R
10,2026-09-27,79.8875,AK,chance,Likely D
15,2026-09-27,0.1250,AR,chance,Safe R
20,2026-09-27,99.9625,CO,chance,Safe D


In [167]:
## Alert when there's a rating change
today_date = rating_ts['date'].max()
yesterday = rating_ts[rating_ts['date'] != today_date]['date'].max()

today_ratings = rating_ts[rating_ts['date'] == today_date]
yester_ratings = rating_ts[rating_ts['date'] == yesterday]
compare_df = pd.merge(left=today_ratings, right=yester_ratings, on='geo', how='left', suffixes=['_t', '_y'])
compare_df.head()

,date_t,y_t,geo,type_t,rating_t,date_y,y_y,type_y,rating_y
0,2026-10-31,62.2000,US Senate,chance,Tilt D,2026-10-30,61.7000,chance,Tilt D
1,2026-10-31,0.5750,AL,chance,Safe R,2026-10-30,0.6125,chance,Safe R
2,2026-10-31,82.7625,AK,chance,Likely D,2026-10-30,83.7375,chance,Likely D
3,2026-10-31,0.1750,AR,chance,Safe R,2026-10-30,0.2125,chance,Safe R
4,2026-10-31,99.9500,CO,chance,Safe D,2026-10-30,99.9625,chance,Safe D


In [168]:
## This is the cell that actually alerts when there's a rating change
if any(compare_df['rating_t'] != compare_df['rating_y']):
    changes_df = compare_df[compare_df['rating_t'] != compare_df['rating_y']][['geo', 'rating_y', 'rating_t']]
    print(changes_df)
else:
    print("No rating changes today!")

No rating changes today!


In [169]:
for i in range(changes_df.shape[0]):
    seat = changes_df.iloc[i, :]['geo']
    rating_y = changes_df.iloc[i, :]['rating_y']
    rating_t = changes_df.iloc[i, :]['rating_t']
    print(f'{seat}: {rating_y} --> {rating_t}')

KS: Tilt R --> Tossup


In [170]:
if any(compare_df['rating_t'] != compare_df['rating_y']):
    print(changes_df.shape)

In [171]:
## Mini-widget: what is the chance of Dems/Reps winning Senate if X state is won by Dems/Reps?
def det_winner(post, state, dem_winner=True):
    '''
    :param post: Posterior simulation matrix.
    :param state: Congressional district.
    :param dem_winner: Whether or not we set Dems or Reps to win `state`. True if Dem win is fixed, False if Rep win is fixed.
    '''
    winners = []
    seats = []
    post_ut = post.T
    if dem_winner:
        post_ut = post_ut[post_ut[state] > 0]
    else:
        post_ut = post_ut[post_ut[state] < 0]
    post_case = post_ut.T
    for i in range(post_case.shape[0]):
        sim = post_case.iloc[:, i]
        dem_seats = np.sum(sim > 0)
        if dem_seats >= 51 - dem_uncont: # Democrats win House in this draw
            winners.append(1)
        else: # Republicans win House in this draw
            winners.append(0)
        seats.append(dem_seats)
    dem_win_chance = np.mean(np.array(winners)) * 100
    dem_mean_seats = np.mean(np.array(seats)) + dem_uncont
    return dem_win_chance, dem_mean_seats

st = 'TX'
dem_winner = True
cond_win_chance, cond_seats = det_winner(post, st, dem_winner)
print(f'If Dems {'win' if dem_winner else 'lose'} {st}-SEN, they have a {cond_win_chance:.1f}% chance of winning the chamber overall, with mean projected seats of {cond_seats:.1f}')

If Dems win TX-SEN, they have a 82.9% chance of winning the chamber overall, with mean projected seats of 53.2


In [172]:
disp_data.to_csv('display_data/choropleth_display_data.csv')
tab_data.to_csv('display_data/table_display_data.csv')
data.to_csv('display_data/all_data.csv')